# 1. Import Libraries

In [ ]:
import os
import sys
import json
import numpy as np
import tensorflow as tf

# 2. File Paths

In [ ]:
scripts_path = os.path.abspath(os.path.join('..', 'Scripts'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

data_dir = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
save_dir = os.path.abspath(os.path.join('..', 'Data', 'SavedModels'))
hyperparameters_dir = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))

# 3. Load Models & Data

## 1. Data

### 1. Train Data

In [ ]:
train_data = np.load(os.path.join(data_dir, 'train_data.npy')).astype(np.float32)

num_users, num_items = train_data.shape
print(f"Dimensi Data: {num_users} Users, {num_items} Items")

### 2. Test Data

In [ ]:
test_data = np.load(os.path.join(data_dir, 'test_data.npy')).astype(np.float32)
test_indices = np.where(test_data > 0)

#### 1. Denormalize Test Data

In [ ]:
MAX_RATING = 5.0
# Denormalisasi nilai asli (karena data di-scaling dengan dibagi 5.0 di awal)
actual_values = test_data[test_indices] * MAX_RATING

## 2. Models

In [ ]:
from model import Encoder, Decoder, VAE

# 4. Construct Model

## 1. VAE

In [ ]:
vae_progress_file = os.path.join(hyperparameters_dir, 'tuning_progress_vae.json')
with open(vae_progress_file, 'r') as f:
    best_vae_params = json.load(f)['best_params']

# Membangun Arsitektur VAE
encoder = Encoder(hidden_dims=best_vae_params['hidden_dims'], latent_dim=best_vae_params['latent_dim'], dropout_rate=best_vae_params['dropout_rate'])
decoder = Decoder(hidden_dims=best_vae_params['hidden_dims'][::-1], output_dim=num_items)
eval_vae = VAE(encoder, decoder, beta=best_vae_params['beta'])

# Menyuntikkan Bobot ke VAE
_ = eval_vae(train_data[:1]) 
eval_vae.load_weights(os.path.join(save_dir, 'trained_best_vae_weights.weights.h5'))

## 2. RSVD

In [ ]:
# Memuat komponen Bias
mu = np.load(os.path.join(save_dir, 'final_mu.npy'))
b_u = np.load(os.path.join(save_dir, 'final_b_u.npy'))
b_i = np.load(os.path.join(save_dir, 'final_b_i.npy'))

# Memuat komponen Laten
U = np.load(os.path.join(save_dir, 'final_U.npy'))
Sigma = np.load(os.path.join(save_dir, 'final_Sigma.npy'))
V = np.load(os.path.join(save_dir, 'final_V.npy'))

# 5. Prediction

## 1. VAE

### 1. Extract Latent Space

In [ ]:
train_data_tf = tf.constant(train_data, dtype=tf.float32)
Z_mean, Z_log_var = eval_vae.encoder.predict(train_data_tf, verbose=0)

### 2. Get Prediction

In [ ]:
# Prediksi dari Decoder (skala 0 - 1)
pred_vae_norm = eval_vae.decoder.predict(Z_mean, verbose=0)

### 3. Filter Prediction
    Filter only test data

In [ ]:
pred_vae_test = pred_vae_norm[test_indices]

### 3. Denormalize

In [ ]:
pred_vae_test_denorm = pred_vae_test * MAX_RATING

### 4. Clip Prediction 

In [ ]:
pred_vae_clipped = np.clip(pred_vae_test_denorm, 1.0, 5.0)

## 2. RSVD

### 1. Latent Dot Product

In [ ]:
latent_matrix = np.dot(np.dot(U, Sigma), V.T)

### 2. Bias Matrix

In [ ]:
bias_matrix = float(mu) + b_u[:, np.newaxis] + b_i[np.newaxis, :]

### 3. Combine

In [ ]:
full_rsvd_pred = bias_matrix + latent_matrix

### 4. Filter Prediction

In [ ]:
pred_rsvd_test = full_rsvd_pred[test_indices]

### 5. Denormalize

In [ ]:
pred_rsvd_test_denorm = pred_rsvd_test * MAX_RATING

### 6. Clip Prediction

In [ ]:
pred_rsvd_clipped = np.clip(pred_rsvd_test_denorm, 1.0, 5.0)

## 3. Ensemble

### 1. Weights

In [ ]:
alpha = 0.5 # Weight for VAE
beta = 0.5  # Weight for RSVD

### 2. Combine

In [ ]:
pred_hybrid = (alpha * pred_vae_test) + (beta * pred_rsvd_test)

### 3. Clip Prediction

In [ ]:
pred_hybrid_clipped = np.clip(pred_hybrid, 1.0, 5.0)

# 6. Evaluation

## 1. VAE

In [ ]:
mse_vae = np.mean(np.square(actual_values - pred_vae_clipped))
rmse_vae = np.sqrt(mse_vae)
mae_vae = np.mean(np.abs(actual_values - pred_vae_clipped))

## 2. RSVD

In [ ]:
mse_rsvd = np.mean(np.square(actual_values - pred_rsvd_clipped))
rmse_rsvd = np.sqrt(mse_rsvd)
mae_rsvd = np.mean(np.abs(actual_values - pred_rsvd_clipped))

## 3. Ensemble

In [ ]:
mse_hybrid = np.mean(np.square(actual_values - pred_hybrid_clipped))
rmse_hybrid = np.sqrt(mse_hybrid)
mae_hybrid = np.mean(np.abs(actual_values - pred_hybrid_clipped))

## 4. Comparison

In [ ]:
print("================================================================")
print("         PERBANDINGAN PERFORMA (ENSEMBLE HYBRID)                ")
print("================================================================")
print(f"Metrik  | VAE Murni   | RSVD Murni  | Ensemble (VAE + RSVD) ")
print("--------|-------------|-------------|-----------------------")
print(f"MSE     | {mse_vae:.4f}      | {mse_rsvd:.4f}      | {mse_hybrid:.4f}")
print(f"RMSE    | {rmse_vae:.4f}      | {rmse_rsvd:.4f}      | {rmse_hybrid:.4f}")
print(f"MAE     | {mae_vae:.4f}      | {mae_rsvd:.4f}      | {mae_hybrid:.4f}")
print("================================================================")

if rmse_hybrid < rmse_vae and rmse_hybrid < rmse_rsvd:
    print("\nKESIMPULAN: LUAR BIASA! Model Hybrid mengalahkan model VAE dan RSVD murni.")
else:
    print("\nKESIMPULAN: Hybrid belum optimal. Cobalah mengatur variabel 'alpha' dan 'beta'.")